<a href="https://colab.research.google.com/github/fourmansyah/Mapperatorinator/blob/main/colab/mapperatorinator_inference_with_video_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beatmap Generation with Mapperatorinator [MOD]

This notebook is an better interactive demo of an osu! beatmap generation model created by OliBomby [MOD]. This model is capable of generating hit objects, hitsounds, timing, kiai times, and SVs for all 4 gamemodes and can do batch osu file. You can upload a beatmap to give to the model as additional context or remap parts of the beatmap.

### Instructions for running:

* Read and accept the rules regarding using this tool by clicking the checkbox.
* Make sure to use a GPU runtime, click:  __Runtime >> Change Runtime Type >> GPU__
* __Execute each cell in order__. Press ▶️ on the left of each cell to execute the cell.
* __Setup Environment__: run the first cell to clone the repository and install the required dependencies. You only need to run this cell once per session.
* __Upload Audio__: choose a .mp3 or .ogg file from your computer.
* __Upload Beatmap__: optionally choose a beatmap .osu file from your computer.  You can find these files in stable by using File > Open Song Folder, or in lazer by using File > Edit Externally.
* __Configure__: choose your generation parameters to control the style of the generated beatmap.
* Generate the beatmap using the __Generate Beatmap__ cell. (it may take a few minutes depending on the length of the song)


In [ ]:
#@title 🚀 Setup Environment { display-mode: "form" }
#@markdown ### Use this tool responsibly. Always disclose the use of AI in your beatmaps. Accept the rules and run this cell.
i_accept_the_rules = False # @param {type:"boolean"}

assert i_accept_the_rules, "Read and accept the rules first!"

!git clone https://github.com/fourmansyah/Mapperatorinator.git
%cd Mapperatorinator

!pip install transformers==4.57.3
!pip install hydra-core nnaudio
!pip install slider git+https://github.com/llllllllll/slider.git
!pip install rosu-pp-py git+https://github.com/MaxOhn/rosu-pp-py.git
!pip install yt-dlp
from google.colab import files

import os
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main

output_path = "output"
input_audio = ""
input_beatmap = ""

In [ ]:
#@title Local Upload Audio { display-mode: "form" }
#@markdown Run this cell to upload audio. This is the song to generate a beatmap for. Please upload a .mp3 or .ogg file.

def upload_audio():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.mp3') and not file.endswith('.ogg'):
        print('Invalid file format. Please upload a .mp3 or .ogg file.')
        return ""
    return data[0]

input_audio = upload_audio()

In [ ]:
#@title 🎵 Download Audio via URL { display-mode: "form" }
#@markdown Enter the YouTube URL or direct link to the audio file. The system will automatically download and display the music player.
audio_url = "" #@param {type:"string"}

import os
import re
import subprocess
from IPython.display import Audio, display

def download_and_validate_audio(url):
    """Download audio from URL with proper error handling and audio player"""
    global audio_path

    if not url or url.strip() == "":
        print("❌ URL kosong! Harap masukkan link yang valid.")
        return None

    print(f"⏳ Mendownload audio dari: {url} ...")

    # Kita buat nama file bersih agar tidak ada karakter aneh dari judul YouTube
    clean_filename = "audio"

    # Hapus file lama agar tidak tertumpuk
    for ext in ['.mp3', '.ogg', '.wav']:
        if os.path.exists(f"{clean_filename}{ext}"):
            os.remove(f"{clean_filename}{ext}")

    try:
        # Perintah yt-dlp untuk ekstrak audio MP3
        command = [
            "yt-dlp",
            "-x",
            "--audio-format", "mp3",
            "--audio-quality", "0",
            "-o", f"{clean_filename}.%(ext)s",
            url
        ]

        # Menjalankan perintah secara diam-diam agar tampilan Colab tetap rapi
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

        # Path file hasil download
        audio_path = os.path.abspath(f"{clean_filename}.mp3")

        if os.path.exists(audio_path):
            print(f"✅ Audio berhasil didownload!")
            print(f"Saved as: {clean_filename}.mp3")
            print(f"Path: {audio_path}")

            # Memunculkan audio player seperti di kode dasar milikmu
            display(Audio(audio_path))

            return audio_path
        else:
            print("❌ File gagal ditemukan setelah proses download selesai.")
            return None

    except subprocess.CalledProcessError:
        print("❌ Terjadi kesalahan saat mendownload. Pastikan URL valid dan video tidak diprivat.")
        return None
    except Exception as e:
        print(f"❌ Error sistem: {e}")
        return None

# Eksekusi fungsi download
audio_path = download_and_validate_audio(audio_url)

# Meneruskan variabel ke input_audio agar Segmen 3 (AI Generator) bisa membacanya
input_audio = audio_path

if not audio_path:
    print("\n⚠️ Silakan periksa kembali URL kamu dan jalankan ulang cell ini.")

In [ ]:
#@title (Optional) Upload Beatmap { display-mode: "form" }
#@markdown This step is required if you want to use `in_context` or `add_to_beatmap` to provide additional info to the model.
#@markdown It will also fill in any missing metadata and unknown values in the configuration using info of the reference beatmap.
#@markdown Please upload a **.osu** file. You can find the .osu file in the song folder in stable or by using File > Edit Externally in lazer.
use_reference_beatmap = False # @param {type:"boolean"}

def upload_beatmap():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.osu'):
        print('Invalid file format. Please upload a .osu file.\nIn stable you can find the .osu file in the song folder (File > Open Song Folder).\nIn lazer you can find the .osu file by using File > Edit Externally.')
        return ""
    return file

if use_reference_beatmap:
    input_beatmap = upload_beatmap()
else:
    input_beatmap = ""

In [ ]:
#@title 🎥 Optional Video Manager { display-mode: "form" }
video_mode = "Disable"  # @param ["Disable", "Download + Inject", "Inject Local Video Only", "Download Only"]
#@markdown - Disable = if you don't want to add video
#@markdown - Download + Inject = if you want to add videos to beatmaps and download the resulting videos.
#@markdown - Inject Local Video Only = if you just want to add videos to beatmaps.
#@markdown - Download Only = if you only want to download the video results.

# Isi salah satu sesuai mode:
#@markdown Youtube Share URL
youtube_video_url = ""  # @param {type:"string"}
#@markdown Colab / Google Drive Path [/content/drive/MyDrive/VideoName.mp4]
video_file_path = ""    # @param {type:"string"}

#@markdown #### Video Setting
video_offset = 0  # @param {type:"integer"}
video_target_height = 720  # @param [480, 720] {type:"raw"}
video_crf = 23  # @param {type:"slider", min:18, max:30, step:1}
video_preset = "medium"  # @param ["veryfast", "fast", "medium", "slow"]
video_output_name = "beatmap_video"  # @param {type:"string"}

import os
import re
import shutil
import subprocess
from google.colab import files

prepared_video_path = None
video_should_inject = False

def _run(cmd):
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True)

def _safe_name(name: str) -> str:
    name = re.sub(r'[\\/*?:"<>|]+', "_", name).strip()
    return name or "beatmap_video"

def inject_video_event(osu_path, video_filename, video_offset=0):
    if not video_filename:
        return

    video_basename = os.path.basename(video_filename)

    with open(osu_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    events_index = None
    for i, line in enumerate(lines):
        if line.strip() == "[Events]":
            events_index = i
            break

    if events_index is None:
        print(f"[WARN] [Events] section tidak ditemukan di {osu_path}")
        return

    for line in lines[events_index + 1:]:
        stripped = line.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            break
        if stripped.startswith("Video,") or stripped.startswith("1,"):
            print(f"[INFO] Video event sudah ada di {osu_path}, skip.")
            return

    insert_index = events_index + 1
    while insert_index < len(lines):
        stripped = lines[insert_index].strip()
        if stripped.startswith("//") or stripped == "":
            insert_index += 1
        else:
            break

    video_line = f'Video,{video_offset},"{video_basename}",0,0\n'
    lines.insert(insert_index, video_line)

    with open(osu_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

    print(f"✅ Video injected into: {osu_path}")

if video_mode == "Disable":
    print("ℹ️ Video feature disabled.")

else:
    if shutil.which("yt-dlp") is None and video_mode in ["Download + Inject", "Download Only"]:
        !pip -q install -U yt-dlp
    if shutil.which("ffmpeg") is None:
        !apt -qq update
        !apt -qq install -y ffmpeg

    os.makedirs("/content/video_tmp", exist_ok=True)
    out_name = _safe_name(video_output_name)
    encoded_output = f"/content/{out_name}.mp4"

    if video_mode in ["Download + Inject", "Download Only"]:
        if not youtube_video_url.strip():
            raise ValueError("Isi youtube_video_url untuk mode download.")

        template = "/content/video_tmp/%(title)s.%(ext)s"
        _run([
            "yt-dlp",
            "-f", "bv*+ba/b",
            "--merge-output-format", "mp4",
            "-o", template,
            youtube_video_url.strip()
        ])

        candidates = []
        for root, _, files_ in os.walk("/content/video_tmp"):
            for f in files_:
                if f.lower().endswith((".mp4", ".mkv", ".webm", ".mov")):
                    candidates.append(os.path.join(root, f))

        if not candidates:
            raise FileNotFoundError("Tidak menemukan video hasil download.")

        source_video = max(candidates, key=os.path.getmtime)

        vf = f"scale='if(gt(ih,{video_target_height}),-2,iw)':'if(gt(ih,{video_target_height}),{video_target_height},ih)'"

        _run([
            "ffmpeg", "-y",
            "-i", source_video,
            "-an",
            "-vf", vf,
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-preset", video_preset,
            "-crf", str(video_crf),
            encoded_output
        ])

        prepared_video_path = encoded_output
        print("✅ Prepared video:", prepared_video_path)

        if video_mode == "Download Only":
            files.download(prepared_video_path)
            print("⬇️ Video downloaded to your laptop.")
        else:
            video_should_inject = True
            print("🎬 Video ready for injection after beatmap generation.")

    elif video_mode == "Inject Local Video Only":
        if not video_file_path.strip():
            raise ValueError("Isi video_file_path untuk mode local inject.")
        if not os.path.exists(video_file_path.strip()):
            raise FileNotFoundError(f"File tidak ditemukan: {video_file_path}")

        prepared_video_path = video_file_path.strip()
        video_should_inject = True
        print("🎬 Local video ready for injection:", prepared_video_path)


In [ ]:
#@title ⚙️ Manual Configure and Generate Beatmap { display-mode: "form" }

#@markdown #### You can input -1 (or leave blank for text) to leave the value unknown.
#@markdown ---
#@markdown This is the AI model to use. V30 is the most accurate model, but it does not support other gamemodes, year, descriptors, or in_context.
model = "Mapperatorinator V30" # @param ["Mapperatorinator V29", "Mapperatorinator V30", "Mapperatorinator V31"]
#@markdown This is the game mode to generate a beatmap for.
gamemode = "standard" # @param ["standard", "taiko", "catch the beat", "mania"]

#@markdown #### Metadata & Map Settings
title = "" # @param {type:"string"}
artist = "" # @param {type:"string"}
creator = "XX_li7fg" # @param {type:"string"}
version_name = "" # @param {type:"string"}
tags = "" # @param {type:"string"}
background = "" # @param {type:"string"}
preview_time = -1 # @param {type:"integer"}

#@markdown #### Style & Difficulty
difficulty = 5 # @param {type:"number"}
#@markdown This is the Star Rating you want your beatmap to be. It might deviate from this number depending on the song intensity and other configuration.
mapper_id = -1 # @param {type:"integer"}
#@markdown This is the user ID of the ranked mapper to imitate for mapping style. You can find this in the URL of the mapper's profile.
beatmap_id = -1 # @param {type:"integer"}
#@markdown This is the Beatmaps ID of the ranked mapper to imitate for mapping style. You can find this in the URL of the Beatmaps List.
year = 2023 # @param {type:"integer"}
#@markdown This is the year you want the beatmap to be from. It should be in the range of 2007 to 2023.
hitsounded = True # @param {type:"boolean"}
#@markdown #### Use Lora if you use v30 to replace Descriptors (Contact Me if you need HF_TOKEN)
lora_path = "" # @param ["fourmansyah/LoRA-2025", "fourmansyah/LoRA-sliderSlop", "fourmansyah/LoRA-aim-control", "fourmansyah/LoRA-Arles-v1", "fourmansyah/LoRA-Kroytz"] {allow-input: true}
#@markdown #### Base Difficulty Settings (Editor)
hp_drain_rate = 5 # @param {type:"number"}
circle_size = 4 # @param {type:"number"}
overall_difficulty = 8 # @param {type:"number"}
approach_rate = 9 # @param {type:"number"}
slider_multiplier = 1.4 # @param {type:"slider", min:0.4, max:3.6, step:0.1}
slider_tick_rate = 1 # @param {type:"number"}

#@markdown #### Mania & Taiko Specifics
keycount = 4 # @param {type:"slider", min:1, max:18, step:1}
hold_note_ratio = -1 # @param {type:"number"}
scroll_speed_ratio = -1 # @param {type:"number"}

#@markdown #### Descriptors (Use v31 because v30 not support)
descriptor_1 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
descriptor_2 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
descriptor_3 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
#@markdown These descriptors of the beatmap. Descriptors are used to describe the style of the beatmap. All available descriptors can be found [here](https://osu.ppy.sh/wiki/en/Beatmap/Beatmap_tags).
negative_descriptor_1 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
negative_descriptor_2 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
negative_descriptor_3 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
#@markdown These are negative descriptors of the beatmap. Negative descriptors are used to describe what the beatmap should not have. These work only when `cfg_scale` is greater than 1.

#@markdown #### Generation Bounds & Context
start_time = -1 # @param {type:"integer"}
end_time = -1 # @param {type:"integer"}
in_context = "[NONE]" # @param ["[NONE]", "[TIMING]", "[TIMING,KIAI]", "[TIMING,KIAI,MAP]", "[GD,TIMING,KIAI]", "[NO_HS,TIMING,KIAI]"]
output_type = "[MAP]" # @param ["[MAP]", "[TIMING,KIAI,MAP,SV]", "[TIMING]"]
add_to_beatmap = False # @param {type:"boolean"}
#@markdown This is which additional information to give to the model:
#@markdown - TIMING: Give timing points to the model. This will skip the timing point generation step.
#@markdown - KIAI: Give kiai times to the model. This will skip the kiai time generation step.
#@markdown - MAP: Give hit objects to the model. This will skip the hit object generation step.
#@markdown - GD: Give hit objects of another difficulty in the same mapset to the model (can be a different game mode). It will improve the rhythm accuracy and consistency of the generated beatmap without copying the reference beatmap.
#@markdown - NO_HS: Give hit objects without hitsounds to the model. This will copy the hit objects of the reference beatmap and only add hitsounds to them.
seed = -1 # @param {type:"integer"}
#@markdown This is the random seed. Change this to sample a different beatmap with the same settings.

#@markdown #### ⏳ Timing & BPM
#@markdown If true, uses a slow and accurate timing generator. This will make the generation slower, but the timing will be more accurate.
timing_leniency = 20 # @param {type:"slider", min:0, max:100, step:1}
#@markdown This is the leniency of the normal timing generator. It will allow the timing ticks to deviate from the real timing by this many milliseconds. A higher value will result in less timing points.
timer_num_beams = 2 # @param {type:"slider", min:1, max:16, step:1}
#@markdown This is the number of beams for beam search for the super timing generator. Higher values will result in slightly more accurate timing at the cost of speed.
timer_iterations = 20 # @param {type:"slider", min:1, max:100, step:1}
#@markdown This is the number of iterations for the super timing generator. Higher values will result in slightly more accurate timing at the cost of speed.
timer_bpm_threshold = 0.1 # @param {type:"slider", min:0, max:1, step:0.1}
#@markdown This is the certainty threshold requirement for BPM changes in the super timing generator. Higher values will result in less BPM changes.
super_timing = False # @param {type:"boolean"}
bpm = -1 # @param {type:"number"}

#@markdown #### Advanced Timing
cfg_scale = 1 # @param {type:"slider", min:1, max:5, step:0.1}
temperature = 1 # @param {type:"slider", min:0, max:1, step:0.01}
timer_cfg_scale = 1.0 # @param {type:"slider", min:0.5, max:5.0, step:0.1}
timing_temperature = 0.1 # @param {type:"slider", min:0.01, max:1.0, step:0.01}
offset = -1 # @param {type:"integer"}
resnap_events = True # @param {type:"boolean"}

#@markdown #### Advanced Sampling
#@markdown #### 0.85–0.90 (Safe) | 0.92–0.95 (Balance) | 0.97–1.00 (Creative)
top_p = 0.95 # @param {type:"slider", min:0.5, max:1.0, step:0.01}
#@markdown #### 10–30 (few top candidates) | 40–100 (more options) | 0 (Ai decide)
top_k = 0 # @param {type:"integer"}
#@markdown #### 1 (Safe) | 2 (Balance) | 3-4 (could be neater, but slower)
num_beams = 1 # @param {type:"integer"}
do_sample = True # @param {type:"boolean"}
#@markdown #### 0.0 (Safe) | Positif (encourage specific timeshift options) | Negatif (reduce that tendency)
timeshift_bias = 0.0 # @param {type:"number"}

#@markdown #### Advanced Diffusion / Positions
diff_cfg_scale = 1.0 # @param {type:"slider", min:0.5, max:5.0, step:0.1}
refine_iters = 10 # @param {type:"slider", min:0, max:30, step:1}
random_init = False # @param {type:"boolean"}

#@markdown #### Long-map Consistency
lookback = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
lookahead = 0.4 # @param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown #### 🔢 Batch Processing (Quantity Control)
#@markdown How many map variations do you want to create at once?
quantity = 1 # @param {type:"slider", min:1, max:10, step:1}
#@markdown #### 📦 Output Packaging
#@markdown Choose how you want to download the results:
download_mode = "Individual .osu Files" # @param ["Master .osz (Lagu + Semua Map)", "Individual .osu Files"]
#@markdown ---

import os
import random
import glob
import zipfile
import shutil
import time
import traceback
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main
from google.colab import files

# Get actual parameters
a_config = model.split(' ')[-1].lower()
a_gamemode = ["standard", "taiko", "catch the beat", "mania"].index(gamemode)
a_title = None if title == "" else title
a_artist = None if artist == "" else artist
a_creator = None if creator == "" else creator
a_base_version = None if version_name == "" else version_name

a_tags = None if tags == "" else tags
a_background = None if background == "" else background
a_preview_time = None if preview_time == -1 else preview_time

a_difficulty = None if difficulty == -1 else difficulty
a_mapper_id = None if mapper_id == -1 else mapper_id
a_beatmap_id = None if beatmap_id == -1 else beatmap_id
a_year = None if year == -1 else year
a_lora_path = None if lora_path == "" else lora_path

a_hp_drain_rate = None if hp_drain_rate == -1 else hp_drain_rate
a_circle_size = None if circle_size == -1 else circle_size
a_overall_difficulty = None if overall_difficulty == -1 else overall_difficulty
a_approach_rate = None if approach_rate == -1 else approach_rate
a_slider_multiplier = None if slider_multiplier == -1 else slider_multiplier
a_slider_tick_rate = None if slider_tick_rate == -1 else slider_tick_rate
a_hold_note_ratio = None if hold_note_ratio == -1 else hold_note_ratio
a_scroll_speed_ratio = None if scroll_speed_ratio == -1 else scroll_speed_ratio

descriptors = [d for d in [descriptor_1, descriptor_2, descriptor_3] if d != '']
negative_descriptors = [d for d in [negative_descriptor_1, negative_descriptor_2, negative_descriptor_3] if d != '']

a_start_time = None if start_time == -1 else start_time
a_end_time = None if end_time == -1 else end_time
a_in_context = [ContextType(c.lower()) for c in in_context[1:-1].split(',')]
a_output_type = [ContextType(c.lower()) for c in output_type[1:-1].split(',')]
a_seed = None if seed == -1 else seed
a_bpm = None if bpm == -1 else bpm

# Validate stuff
if any(c in a_in_context for c in [ContextType.TIMING, ContextType.KIAI, ContextType.MAP, ContextType.SV, ContextType.GD, ContextType.NO_HS]) or add_to_beatmap:
    assert os.path.exists(input_beatmap), "Please upload a reference beatmap."
assert os.path.exists(input_audio), "Please upload an audio file."

if a_config == "v30":
    assert a_gamemode == 0, "V30 only supports standard mode."
    if any(c in a_in_context for c in [ContextType.KIAI, ContextType.MAP, ContextType.SV]):
        print("WARNING: V30 does not support KIAI, MAP, or SV in_context, ignoring.")
    if output_type != "[MAP]":
        print("WARNING: V30 only supports [MAP] output type, setting output type to [MAP].")
        a_output_type = [ContextType.MAP]
    if len(descriptors) != 0 or len(negative_descriptors) != 0:
        print("WARNING: V30 does not support descriptors or negative descriptors, ignoring.")
    if super_timing:
        print("WARNING: V30 does not fully support super timing, generation will be VERY slow.")

# Bersihkan output folder lama
if os.path.exists(output_path):
    shutil.rmtree(output_path)
os.makedirs(output_path, exist_ok=True)

generated_osu_files = []

print(f"🚀 Memulai pembuatan {quantity} beatmap untuk Quality Control...")
print("==================================================")

# Looping Generate berdasarkan Quantity
for i in range(quantity):
    take_num = i + 1

    # Pengaturan Nama Version otomatis
    current_version = f"{a_base_version} (Take {take_num})" if quantity > 1 and a_base_version else a_base_version
    if not current_version:
        current_version = f"AI Generated (Take {take_num})"

    # Pengaturan Seed Acak/Urut
    current_seed = random.randint(1, 999999) if a_seed is None else (a_seed + i)

    print(f"\n🎬 Proses Take {take_num}/{quantity} | Diff: {current_version} | Seed: {current_seed}")

    try:
        # Bersihkan cache hydra agar tidak error saat dilooping
        try:
            from hydra.core.global_hydra import GlobalHydra
            GlobalHydra.instance().clear()
        except:
            pass

        # Create config
        with initialize_config_dir(version_base="1.1", config_dir="/content/Mapperatorinator/configs/inference"):
            conf = compose(config_name=a_config)

        # Map all attributes to conf object
        conf.audio_path = input_audio
        conf.output_path = output_path
        conf.beatmap_path = input_beatmap if 'input_beatmap' in globals() else None

        # Apply Metadata & Map Settings
        conf.gamemode = a_gamemode
        conf.title = a_title
        conf.artist = a_artist
        conf.creator = a_creator
        conf.version = current_version
        conf.tags = a_tags
        conf.background = a_background
        conf.preview_time = a_preview_time

        # Apply Style & Difficulty
        conf.difficulty = a_difficulty
        conf.mapper_id = a_mapper_id
        conf.beatmap_id = a_beatmap_id
        conf.year = a_year
        conf.lora_path = a_lora_path
        conf.hitsounded = hitsounded

        # Apply Editor Stats
        conf.hp_drain_rate = a_hp_drain_rate
        conf.circle_size = a_circle_size
        conf.overall_difficulty = a_overall_difficulty
        conf.approach_rate = a_approach_rate
        conf.slider_multiplier = a_slider_multiplier
        conf.slider_tick_rate = a_slider_tick_rate
        conf.keycount = keycount
        conf.hold_note_ratio = a_hold_note_ratio
        conf.scroll_speed_ratio = a_scroll_speed_ratio

        # Apply Descriptors & Bounds
        conf.descriptors = descriptors
        conf.negative_descriptors = negative_descriptors

        # Force export_osz False di dalam config karena kita akan membuat Master OSZ sendiri di akhir
        conf.export_osz = False
        conf.add_to_beatmap = add_to_beatmap
        conf.start_time = a_start_time
        conf.end_time = a_end_time
        conf.in_context = a_in_context
        conf.output_type = a_output_type
        conf.cfg_scale = cfg_scale
        conf.temperature = temperature
        conf.seed = current_seed

        # Apply Timing
        conf.timing_leniency = timing_leniency
        conf.super_timing = super_timing
        conf.timer_num_beams = timer_num_beams
        conf.timer_iterations = timer_iterations
        conf.timer_bpm_threshold = timer_bpm_threshold
        conf.bpm = a_bpm

        # Advanced Sampling
        conf.top_p = top_p
        conf.top_k = 20 if top_k is None or top_k <= 0 else top_k
        conf.num_beams = max(1, num_beams)
        conf.do_sample = do_sample
        conf.timeshift_bias = timeshift_bias

        conf.timing_temperature = timing_temperature
        conf.timer_cfg_scale = timer_cfg_scale
        conf.offset = None if offset == -1 else offset
        conf.resnap_events = resnap_events

        conf.diff_cfg_scale = diff_cfg_scale
        conf.refine_iters = refine_iters
        conf.random_init = random_init

        conf.lookback = lookback
        conf.lookahead = lookahead

        # Run Inference
        _, result_path, _ = main(conf)

        # Simpan jejak file .osu yang berhasil dibuat
        if result_path and result_path.endswith('.osu'):
            if 'video_should_inject' in globals() and video_should_inject and 'prepared_video_path' in globals() and prepared_video_path:
                inject_video_event(result_path, prepared_video_path, video_offset)

            generated_osu_files.append(result_path)
            print(f"✅ Selesai Take {take_num}!")

    except Exception as e:
        print(f"❌ Terjadi kesalahan pada Take {take_num}: {e}")
        traceback.print_exc()

# ==========================================
# PROSES PENGUNDUHAN (BERDASARKAN PILIHAN)
# ==========================================
if generated_osu_files:
    if download_mode == "Master .osz (Lagu + Semua Map)":
        print("\n📦 Mengemas semua variasi map ke dalam SATU file .osz...")
        file_title = a_title if a_title else "Unknown Title"
        file_artist = a_artist if a_artist else "Unknown Artist"
        master_osz_name = f"{file_artist} - {file_title} (Master Batch).osz"

        with zipfile.ZipFile(master_osz_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(input_audio, os.path.basename(input_audio))
            if a_background and os.path.exists(a_background):
                zipf.write(a_background, os.path.basename(a_background))

            if 'video_should_inject' in globals() and video_should_inject and 'prepared_video_path' in globals() and prepared_video_path and os.path.exists(prepared_video_path):
                zipf.write(prepared_video_path, os.path.basename(prepared_video_path))

            for osu_file in generated_osu_files:
                zipf.write(osu_file, os.path.basename(osu_file))

        print(f"🎉 SANGAT SUKSES! Mengunduh file otomatis: {master_osz_name}")
        files.download(master_osz_name)

    else:
        print(f"\n📄 Mengunduh {len(generated_osu_files)} file .osu satu per satu...")
        for osu_file in generated_osu_files:
            print(f"Memicu unduhan untuk: {os.path.basename(osu_file)}")
            files.download(osu_file)
            time.sleep(1.5) # Jeda agar browser tidak memblokir unduhan beruntun

        print("🎉 SANGAT SUKSES! Semua file .osu berhasil diunduh.")
else:
    print("\n❌ Gagal membungkus/mengunduh: Tidak ada file .osu yang berhasil di-generate.")